<a href="https://colab.research.google.com/github/shravan1808/ML_SERIES/blob/Main/11_API_Driven_Logistic_Regression_Decision_Boundary_Engine/notebook/Project_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
import requests
import io
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score

In [37]:
def fetch_api_data(url):
  response = requests.get(url)
  response.raise_for_status()
  data = pd.read_csv(io.StringIO(response.text))
  return data

In [38]:
FLOW_LOGS_API_ENDPOINT = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"

In [39]:
penguin_df = fetch_api_data(FLOW_LOGS_API_ENDPOINT)

In [40]:
penguin_df.dropna(inplace=True)

In [41]:
penguin_df['Is_Anomalous'] = np.where(penguin_df['species'] == 'Adelie', 0, 1)

In [42]:
penguin_df.drop(['species','island','sex'],inplace=True,axis=1)

In [43]:
penguin_df.shape

(333, 5)

In [44]:
penguin_df['Is_Anomalous'].value_counts()

,count
Is_Anomalous,
1,187
0,146


In [45]:
penguin_df.head(3)

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,Is_Anomalous
0,39.1,18.7,181.0,3750.0,0
1,39.5,17.4,186.0,3800.0,0
2,40.3,18.0,195.0,3250.0,0


In [46]:
X=penguin_df.drop('Is_Anomalous',axis=1)
y=penguin_df['Is_Anomalous']

In [47]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.3,random_state=42,stratify=y)

In [48]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [49]:
X_train_size = X_train.shape[0]
X_test_size = X_test.shape[0]
y_train_size = y_train.shape[0]
y_test_size = y_test.shape[0]
y_train_counts = y_train.value_counts()
y_test_counts = y_test.value_counts()

In [50]:
model = LogisticRegression(random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

y_prob = model.predict_proba(X_test_scaled)[:, 1]

In [51]:
model_intercept = model.intercept_
model_coefficients = model.coef_

In [52]:
print(model_intercept)

[0.97661427]


In [53]:
for col in X.columns:
  print(f"{col:<20} : {round(model_coefficients[0][X.columns.get_loc(col)],3)}")

bill_length_mm       : 3.799
bill_depth_mm        : -1.872
flipper_length_mm    : 0.445
body_mass_g          : -0.396


In [54]:
for i in range(5):
  if y_prob[i]>0.5:
    print(f"Sample {i+1} | P(Anomalous) : {round(y_prob[i],4)} | Label : {y_test.iloc[i]} | Assigned : High Risk")
  else:
    print(f"Sample {i+1} | P(Anomalous) : {round(y_prob[i],4)} | Label : {y_test.iloc[i]} | Assigned : Low Risk")

Sample 1 | P(Anomalous) : 0.9962 | Label : 1 | Assigned : High Risk
Sample 2 | P(Anomalous) : 0.0117 | Label : 0 | Assigned : Low Risk
Sample 3 | P(Anomalous) : 0.9953 | Label : 1 | Assigned : High Risk
Sample 4 | P(Anomalous) : 0.9975 | Label : 1 | Assigned : High Risk
Sample 5 | P(Anomalous) : 0.9999 | Label : 1 | Assigned : High Risk


In [55]:
thresholds = [0.30, 0.50, 0.70]
for threshold in thresholds:
  y_pred_threshold = (y_prob>=threshold).astype(int)
  accuracy = accuracy_score(y_test, y_pred_threshold)
  precision = precision_score(y_test, y_pred_threshold)
  recall = recall_score(y_test, y_pred_threshold)
  print(f"Threshold: {threshold} | Accuracy: {accuracy*100}% | Precision: {precision:.4f} | Recall: {recall:.4f}")


Threshold: 0.3 | Accuracy: 99.0% | Precision: 0.9825 | Recall: 1.0000
Threshold: 0.5 | Accuracy: 100.0% | Precision: 1.0000 | Recall: 1.0000
Threshold: 0.7 | Accuracy: 100.0% | Precision: 1.0000 | Recall: 1.0000


In [56]:
print("\n========== API-DRIVEN LOGISTIC REGRESSION & DECISION BOUNDARY ENGINE ==========")

print("\nData Ingestion Status      : REST API Ingestion Successful (HTTP 200 OK)")
print(f"Master Dataset Records     : {penguin_df.shape[0]} (Cleaned & Preprocessed)")

print(
    "Features Included          : "
    "4 Continuous Log Metrics "
    "(bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g)"
)

print(
    "Target Output              : "
    "Is_Anomalous "
    "(Binary Classification: 0 = Standard, 1 = Anomalous)"
)

print("\nModel Training Metrics:")
print(
    f"- Stratified Train Split   : {X_train_size} Records "
    f"({y_train_counts[0]} Normal / {y_train_counts[1]} Anomalous)"
)

print(
    f"- Stratified Test Split    : {X_test_size} Records "
    f"({y_test_counts[0]} Normal / {y_test_counts[1]} Anomalous)"
)

base_accuracy = accuracy_score(y_test, y_pred)

print(
    f"- Base Test Accuracy       : "
    f"{base_accuracy * 100:.1f}% (at Default 0.50 Threshold)"
)

# Probability profiling
max_prob = y_prob.max()
min_prob = y_prob.min()

# Find two largest absolute coefficients
coef_series = pd.Series(model.coef_[0], index=X.columns)
top_drivers = coef_series.abs().sort_values(ascending=False).index[:2]

print("\nSigmoid Probability Profiling:")
print(f"- Maximum Anomalous Prob   : {max_prob * 100:.2f}%")
print(f"- Minimum Anomalous Prob   : {min_prob * 100:.2f}%")

print(
    f"- Key Probability Drivers : "
    f"{top_drivers[0]} ({coef_series[top_drivers[0]]:+.4f}), "
    f"{top_drivers[1]} ({coef_series[top_drivers[1]]:+.4f})"
)

# Threshold evaluation
print("\nDecision Threshold Tuning:")

for threshold in thresholds:

    y_pred_threshold = (y_prob >= threshold).astype(int)

    accuracy = accuracy_score(y_test, y_pred_threshold)
    precision = precision_score(y_test, y_pred_threshold)
    recall = recall_score(y_test, y_pred_threshold)

    if threshold == 0.30:
        description = "Strict"
        explanation = "Maximizes detection for suspicious traffic"

    elif threshold == 0.50:
        description = "Normal"
        explanation = "Balanced baseline model performance"

    else:
        description = "Alert"
        explanation = "Minimizes false alarms and alert fatigue"

    print(
        f"- Threshold @ {threshold:.2f} ({description:<6}) : "
        f"{accuracy * 100:.1f}% Accuracy | "
        f"Precision: {precision:.4f} | "
        f"Recall: {recall:.4f} "
        f"({explanation})"
    )

print("\nConclusion:")
print(
    "By implementing dynamic REST API fetching, the data pipeline dynamically "
    "ingests raw security logs over HTTP. Logistic Regression maps continuous "
    "features to calibrated sigmoid probability scores, enabling security "
    "operations teams to tune decision boundaries based on system risk tolerance."
)


========== API-DRIVEN LOGISTIC REGRESSION & DECISION BOUNDARY ENGINE ==========

Data Ingestion Status      : REST API Ingestion Successful (HTTP 200 OK)
Master Dataset Records     : 333 (Cleaned & Preprocessed)
Features Included          : 4 Continuous Log Metrics (bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g)
Target Output              : Is_Anomalous (Binary Classification: 0 = Standard, 1 = Anomalous)

Model Training Metrics:
- Stratified Train Split   : 233 Records (102 Normal / 131 Anomalous)
- Stratified Test Split    : 100 Records (44 Normal / 56 Anomalous)
- Base Test Accuracy       : 100.0% (at Default 0.50 Threshold)

Sigmoid Probability Profiling:
- Maximum Anomalous Prob   : 100.00%
- Minimum Anomalous Prob   : 0.11%
- Key Probability Drivers : bill_length_mm (+3.7992), bill_depth_mm (-1.8720)

Decision Threshold Tuning:
- Threshold @ 0.30 (Strict) : 99.0% Accuracy | Precision: 0.9825 | Recall: 1.0000 (Maximizes detection for suspicious traffic)
- Threshol